# 04 — Masking scenarios (Table 7, S2) and group-level tables (Table 6, S-tables)
Re-evaluates the 12 availability-aware checkpoints (4 variants x 3 seeds) and the training-free baselines (A, B, NeighborMean, Ring2Mean) under 25% validation masking, random masking at 10/25/50% and structural neighbourhood masking (p_centre = 0.05), then rebuilds the group-level (leakage_group_id) tables and compares every regenerated CSV with the file stored during the revision. No training, no tuning. Models were trained only with 25% random masking; the other scenarios are distribution shifts.

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, json, pandas as pd, numpy as np, torch
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability, effective_mask, score_mask
from sasp_gat.evaluation import load_model, eval_both
from sasp_gat.training import evaluate_multimask_av

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
ppi_ei = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order, device)
CKDIR = REPO / 'checkpoints' if (REPO / 'checkpoints' / 'full_avail_seed42.pt').exists() else Path(BASE) / 'revision_checkpoints_2026-10-18'
print('checkpoint dir:', CKDIR)
import torch_geometric, scipy; print('versions: torch', torch.__version__, '| pyg', torch_geometric.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__)
import warnings; warnings.filterwarnings('ignore')
ABL_DIR = Path(BASE) / 'revision_checkpoints_2026-10-18'
TAB_DIR = Path(BASE) / 'revision_tables_2026-10-18'
MR_DIR = Path(BASE) / 'revision_masking_robustness_2026-10-19'
OUT_DIR = Path(BASE) / 'repro_outputs_2026-10-24'   # regenerated tables (never overwrites the revision folders)
OUT_DIR.mkdir(exist_ok=True)


In [ ]:
from sasp_gat.evaluation import load_all_models, VARIANT_KEYS
from sasp_gat.twohop import build_graph_cache
from sasp_gat.baselines import train_gene_means
from sasp_gat.scenarios import Predictors, run_all_scenarios, group_tables
from sasp_gat.verify import compare_csv, report_compare
# full model from the repo; ablation variants from Drive (checkpoints of the revision)
MODELS = load_all_models(CKDIR, device, variants=['full'])
MODELS.update(load_all_models(ABL_DIR, device, variants=[v for v in VARIANT_KEYS if v != 'full']))
assert len(MODELS) == 12
mu32 = train_gene_means(train_s, AVAIL)
GR = build_graph_cache({'val': val_s, 'test': test_s, 'hd': hd_s}, device)
pred = Predictors(MODELS, ppi_ei, mu32, device)
print('models', len(MODELS), '| slides in graph cache', len(GR))

In [ ]:
# Scenarios 17A-17C (several minutes on GPU)
import time; t0 = time.time()
RES17, STAT17, SUM17 = run_all_scenarios(pred, AVAIL, val_s, test_s, hd_s, GR, device)
FILES = group_tables(RES17, SUM17, STAT17)
print('done in %.0f s' % (time.time() - t0), '| rows', len(RES17))
# headline: pooled 25% arm must equal the primary results (full model, test/HD)
s25 = SUM17[('rand25', 'test')]; print(s25.round(5).to_string()); print(SUM17[('rand25', 'hd')].round(5).to_string())

In [ ]:
# Compare every regenerated CSV with the stored one (same file name) in revision_masking_robustness_2026-10-19
print('stored files:', sorted(p.name for p in MR_DIR.glob('*.csv')))
results = {}
for name, df in FILES.items():
    p = MR_DIR / name
    results[name] = compare_csv(df, p) if p.exists() else (False, float('nan'), 'stored file not found under this name')
ALL_OK = report_compare(results)

In [ ]:
# Group-level evaluation: full vs expression+spatial (test and HD; leakage groups and cancer types)
from sasp_gat.group_eval import run_group_eval
GFILES, GR_, GSUMS = run_group_eval(MODELS, test_s, hd_s, AVAIL, ppi_ei, device)
print('stored files in revision_tables:', sorted(p.name for p in TAB_DIR.glob('group*')))
res2 = {}
for name, df in GFILES.items():
    p = TAB_DIR / name
    res2[name] = compare_csv(df, p) if p.exists() else (False, float('nan'), 'stored file not found under this name')
report_compare(res2)
for (coh, lvl), (G, T, O) in GR_.items():
    print(f'\n== {coh} / {lvl}'); print(T.round(4).to_string(index=False))

In [ ]:
# Save regenerated tables for notebook 05 (new folder; existing revision folders are not touched)
for name, df in {**FILES, **GFILES}.items():
    df.to_csv(OUT_DIR / name, index=False)
print('saved', len(FILES) + len(GFILES), 'regenerated tables to', OUT_DIR)